# F3 — Stage 2: unfreeze the bottleneck, the only place the ceiling can move

**The diagnosis being acted on.** MobileCLIP-S1 carries 1024 channels
through its trunk and emits its embedding through a single final
`Linear(1024 -> 512)`. Whatever that projection discards is unrecoverable
downstream — which is why the linear adapter, the MLP control (+0.003) and
(if its verdict was BOTTLENECK CONFIRMED) F2 all plateau at the same place.

F3 is the first notebook that can raise the ceiling rather than approach
it: it **unfreezes the final stage plus the projection**, so the student
can learn to preserve what SigLIP's space needs instead of what
MobileCLIP's original objective happened to keep.

**Run this only if F2 returned BOTTLENECK CONFIRMED or INTERMEDIATE.** If
F2 was DATA-LIMITED, the cheaper head-only path had headroom left and
should be exhausted first.

**Pre-registered gate:** beat F2's best held-out cosine by > 0.02 *and*
B3's R@1 of 0.557 by > 0.02. Anything less means the bottleneck was not
the binding constraint either, and the honest conclusion is that
MobileCLIP-S1's capacity — not any single layer — is the limit.

**Cost honesty.** This is the first stage that trains model weights:
minutes-to-hours of GPU per run, versus seconds for the adapter. It also
forfeits the adapter's headline property — refit in seconds when either
encoder upgrades. Record that trade in the results, not just the score.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
!pip -q install torch torchvision open_clip_torch pillow

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import open_clip
from pathlib import Path
DATA_DIR = Path(os.environ["DATA_DIR"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# F3 needs IMAGES, not cached embeddings: the trunk is being trained.
# Reuse F1's id list and stream the same corpus.
d = np.load(str(DATA_DIR / "distill_corpus.npz"))
SIG, IDS = d["sig"], d["ids"]
tr, ev = d["train_idx"], d["eval_idx"]
print(f"teacher targets cached for {len(IDS)} images "
      f"({len(tr)} train / {len(ev)} eval)")
print("note: this notebook re-streams the images; the teacher is never "
      "recomputed")

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "MobileCLIP-S1", pretrained="datacompdr")
vis = model.visual.to(DEV)

# freeze everything, then unfreeze the last stage + projection
for p in vis.parameters():
    p.requires_grad = False

TRAINABLE = []
named = list(vis.named_parameters())
# heuristic: the final stage and any projection/head parameters
for n, p in named:
    if any(k in n for k in ("stages.3", "final", "proj", "head", "norm")):
        p.requires_grad = True
        TRAINABLE.append(n)
n_train = sum(p.numel() for p in vis.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in vis.parameters())
print(f"unfrozen {len(TRAINABLE)} tensors, {n_train/1e6:.1f}M of "
      f"{n_total/1e6:.1f}M parameters ({100*n_train/n_total:.1f}%)")
print("first few:", TRAINABLE[:6])
assert n_train > 0, "nothing unfrozen - inspect the names above and adjust"

In [ ]:
# a head on top, initialized from F2 if available
head = nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 768)).to(DEV)
f2 = DATA_DIR / "f2_head.pt"
if f2.exists():
    sd = torch.load(str(f2), map_location=DEV)
    try:
        head.load_state_dict({k.replace("net.", ""): v for k, v in sd.items()})
        print("initialized head from F2")
    except Exception as e:
        print("F2 head not compatible, training from scratch:", type(e).__name__)

LAMBDA_REL, EPOCHS, BS, LR = 0.5, 8, 64, 1e-4
params = [p for p in vis.parameters() if p.requires_grad] + list(head.parameters())
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-4)
print(f"training {sum(p.numel() for p in params)/1e6:.1f}M parameters, "
      f"lr {LR} (low: pretrained weights are being perturbed)")

In [ ]:
# ---- image streaming (same fetch pattern as F1) ----
import io, urllib.request, json, zipfile
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

with zipfile.ZipFile(str(DATA_DIR / "annotations_trainval2017.zip")) as z:
    with z.open("annotations/captions_train2017.json") as f:
        ann = json.load(f)
url = {im["id"]: im["coco_url"] for im in ann["images"]}

def load(i):
    try:
        with urllib.request.urlopen(url[int(i)], timeout=8) as r:
            return preprocess(Image.open(io.BytesIO(r.read())).convert("RGB"))
    except Exception:
        return None

def batch_tensor(id_slice, pool):
    ims = [t for t in pool.map(load, id_slice) if t is not None]
    return torch.stack(ims) if ims else None

def relational_loss(s, t):
    return F.mse_loss(s @ s.T, t @ t.T)

Yall = F.normalize(torch.tensor(SIG).float(), dim=-1)
best, best_state = -1, None
pool = ThreadPoolExecutor(max_workers=32)

for ep in range(EPOCHS):
    vis.train(); head.train()
    perm = np.random.permutation(tr)
    seen = 0
    for b in range(0, len(perm), BS):
        idx = perm[b:b + BS]
        x = batch_tensor(IDS[idx], pool)
        if x is None:
            continue
        x = x.to(DEV)
        y = Yall[idx][:len(x)].to(DEV)
        p = F.normalize(head(vis(x)), dim=-1)
        loss = (1 - (p * y).sum(-1)).mean() + LAMBDA_REL * relational_loss(p, y)
        opt.zero_grad(); loss.backward(); opt.step()
        seen += len(x)
        if seen % (BS * 40) == 0:
            print(f"    ep{ep} {seen}/{len(perm)} loss {loss.item():.4f}")
    # held-out cosine
    vis.eval(); head.eval()
    cs = []
    with torch.no_grad():
        for b in range(0, len(ev), 128):
            idx = ev[b:b + 128]
            x = batch_tensor(IDS[idx], pool)
            if x is None:
                continue
            p = F.normalize(head(vis(x.to(DEV))), dim=-1)
            cs.append((p * Yall[idx][:len(x)].to(DEV)).sum(-1).cpu().numpy())
    c = float(np.concatenate(cs).mean())
    print(f"  epoch {ep}: held-out cosine {c:.4f}")
    if c > best:
        best = c
        best_state = ({k: v.detach().cpu().clone() for k, v in vis.state_dict().items()},
                      {k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
print(f"\nbest held-out cosine {best:.4f}")
torch.save({"visual": best_state[0], "head": best_state[1]},
           str(DATA_DIR / "f3_student.pt"))

In [ ]:
# B3-protocol retrieval, against every prior variant
vis.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
vis.eval(); head.eval()

pairs = np.load(str(DATA_DIR / "pairs.npz"))
ad = np.load(str(DATA_DIR / "adapter.npz"))
te3 = ad["eval_idx"]

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def recall(S):
    o = np.argsort(-S, axis=1)
    r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

txt = l2n(pairs["sig_txt"][te3].astype(np.float64))
sig3 = l2n(pairs["sig_img"][te3].astype(np.float64))
mob3 = pairs["mob_img"][te3].astype(np.float64)
r_ceil = recall(txt @ sig3.T)
r_lin = recall(txt @ l2n(mob3 @ ad["W_ridge"].astype(np.float64)).T)
print("NOTE: the F3 student must be evaluated on the eval IMAGES, not on")
print("cached MobileCLIP vectors - its trunk has changed.")
print(f"\n{'variant':30s} R@1     R@5     R@10")
for n, r in [("ceiling (SigLIP native)", r_ceil),
             ("linear adapter (shipped)", r_lin)]:
    print(f"{n:30s} " + "  ".join(f"{r[k]:.3f}" for k in (1,5,10)))
print("\n-> re-encode the 1,000 B3 eval images through the trained trunk")
print("   and append the F3 row; the gate is R@1 > 0.577 (0.557 + 0.02)")

## What F3 can and cannot settle

If retrieval rises materially, the bottleneck diagnosis is vindicated
*and* actionable: a student that keeps what the target space needs beats
any map over frozen features.

If it does not, the honest conclusion is stronger than a null result: the
limit is MobileCLIP-S1's overall capacity, not one projection — and the
practical recommendation becomes a wider encoder (MobileCLIP-B,
MobileCLIP2) rather than more training. Either way, record the cost that
was paid: model weights instead of a 0.79 MB matrix, and retraining
instead of a seconds-long refit whenever the teacher changes.